# UAV Battery Tool — Notebook 05: Report Generation

Generate formatted Excel reports from simulation results, battery comparisons, and flight log analysis.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

from batteries.database import BatteryDatabase
from batteries.log_importer import generate_synthetic_log
from batteries.parameter_fitter import fit_all, apply_fitted_params
from mission.simulator import run_simulation, compare_batteries, temperature_sweep
from mission.report_generator import generate_report

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

═══ Battery Database Summary ═══
  Chemistries       : 9
  Cells             : 11
  Battery packs     : 125
  Discharge points  : 132
  Equipment items   : 29
  UAV configurations: 3
  Mission profiles  : 3


## 1 · Configure Report Parameters

In [2]:
# ── Load settings from 00_configurator.ipynb (analysis_config.json) ───────────
import json as _json, os as _os
# Look for analysis_config.json: try ../  (notebooks/ CWD) then ./  (root CWD)
for _p in ['../analysis_config.json', 'analysis_config.json']:
    if _os.path.exists(_p):
        _CFG_PATH = _p
        break
else:
    _CFG_PATH = '../analysis_config.json'  # default write-path
_cfg = {}
if _os.path.exists(_CFG_PATH):
    with open(_CFG_PATH) as _f:
        _cfg = _json.load(_f)
    print(f'Loaded config from {_CFG_PATH}')
else:
    print('No analysis_config.json found — using defaults below (run 00_configurator first)')

# ── Values from configurator (overridden by manual entries below if desired) ───
_sel_bats = _cfg.get('selected_batteries', ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P'])
PRIMARY_PACK_ID  = _sel_bats[0] if isinstance(_sel_bats, list) and _sel_bats else 'BAT_MID_6S2P'
COMPARE_PACK_IDS = _sel_bats if isinstance(_sel_bats, list) else ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
MISSION_ID       = _cfg.get('mission_id',     'SURVEY_STD')
UAV_ID           = _cfg.get('uav_id',         'HEX_SURVEY_900')
AMBIENT_TEMP_C   = _cfg.get('ambient_temp_c', 25.0)
TEMP_SWEEP       = _cfg.get('temp_sweep',     [-25, -10, 0, 15, 25, 40])


# ── Reconstruct combined pack if configured ────────────────────────────────────
_combo_cfg = _cfg.get('battery_combination')
if _combo_cfg:
    from batteries.builder import combine_packs as _combine_packs
    _combo_packs = [db.packs[bid] for bid in _combo_cfg.get('packs', []) if bid in db.packs]
    if len(_combo_packs) >= 2:
        _combined = _combine_packs(_combo_packs, topology=_combo_cfg.get('topology', 'series'))
        db.packs[_combined.battery_id] = _combined
        print(f'Combined pack registered: {_combined.battery_id}  '
              f'({_combined.pack_voltage_nom:.1f}V {_combined.pack_energy_wh:.0f}Wh)')

# ── Manual overrides (uncomment to override configurator values) ────────────
# ── Select packs, mission, and UAV ────────────────────────────────────────────
# PRIMARY_PACK_ID    = 'BAT_MID_6S2P'
# COMPARE_PACK_IDS   = ['BAT_MID_6S2P', 'BAT_MID_6S4P', 'BAT_AG_6S1P']
# MISSION_ID         = 'SURVEY_STD'
# UAV_ID             = 'HEX_SURVEY_900'
# AMBIENT_TEMP_C     = 25.0
# TEMP_SWEEP         = [-20,-15,-10,-5,0,5,10,15,20,25,30,35,40,45]
INCLUDE_LOG        = True   # generate synthetic log and fit parameters
OUT_PATH           = 'UAV_Battery_Report.xlsx'
# ─────────────────────────────────────────────────────────────────────────────

primary_pack = db.packs[PRIMARY_PACK_ID]
mission      = db.missions[MISSION_ID]
uav          = db.uav_configs[UAV_ID]
compare_packs= [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in db.packs]
print(f'Primary pack  : {primary_pack}')
print(f'Compare packs : {[p.battery_id for p in compare_packs]}')
print(f'Mission       : {mission}')

Loaded config from analysis_config.json
Primary pack  : BAT_LE_12S2P: Long Endurance 12S2P 21700 | 12S2P | 43.8V 8.0Ah (350Wh) | 1656g
Compare packs : ['BAT_LE_12S2P', 'GREPOW_TATTU_12S_LIPO_DRONE_BATTERY', 'GREPOW_TATTU_12S_10000MAH_44_4V_30C_LIPO_BATTERY', 'TATTU_TATTU_PLUS_1_0_COMPACT_VERSION_10000MAH_44_4', 'GREPOW_TATTU_PLUS_1_0_12S_10000MAH_44_4V_15C_LIPO_', 'TATTU_TATTU_SEMI_SOLID_STATE_330WH_KG_25000MAH_10C']
Mission       : Mission 'Grid Survey 500m' (SURVEY_STD): 8 phases | 24.5 min | 6700m


## 2 · Run Simulations

In [3]:
print('Running simulations...')

# Run simulation for ALL selected packs
all_results = {}
for _pid in COMPARE_PACK_IDS:
    if _pid not in db.packs:
        print(f'  [SKIP] {_pid} — not in database')
        continue
    _r = run_simulation(
        pack=db.packs[_pid], mission=mission, uav=uav,
        discharge_pts=db.discharge_pts,
        ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0
    )
    all_results[_pid] = _r
    print(_r.summary())

# Aliases kept for downstream compatibility
primary_result  = all_results.get(PRIMARY_PACK_ID, next(iter(all_results.values())))
compare_results = list(all_results.values())
compare_packs   = [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in all_results]

print(f'\nMulti-pack results ({len(compare_results)} packs):')
for r in compare_results:
    print(f'  {r.pack_id}: SoC={r.final_soc:.1f}%  Vmin={r.min_voltage:.3f}V  '
          f'depleted={r.depleted}')

Running simulations...
════════════════════════════════════════════════════
 Simulation: BAT_LE_12S2P × SURVEY_STD  [COMPLETED]
════════════════════════════════════════════════════
  Duration         : 1469 s  (24.5 min)
  Energy consumed  : 127.91 Wh
  Initial SoC      : 100.0 %
  Final SoC        : 66.6 %
  Min voltage      : 45.078 V
  Max current      : 14.7 A
  Peak sag total   : 0.945 V
  Peak temperature : 25.8 °C
════════════════════════════════════════════════════
 Simulation: GREPOW_TATTU_12S_LIPO_DRONE_BATTERY × SURVEY_STD  [COMPLETED]
════════════════════════════════════════════════════
  Duration         : 1469 s  (24.5 min)
  Energy consumed  : 127.91 Wh
  Initial SoC      : 100.0 %
  Final SoC        : 74.0 %
  Min voltage      : 46.453 V
  Max current      : 14.5 A
  Peak sag total   : 0.347 V
  Peak temperature : 25.4 °C
════════════════════════════════════════════════════
 Simulation: GREPOW_TATTU_12S_10000MAH_44_4V_30C_LIPO_BATTERY × SURVEY_STD  [COMPLETED]
═════════

In [4]:
# Temperature sweep for ALL selected packs
print(f'Running temperature sweeps ({len(TEMP_SWEEP)} temps x {len(compare_packs)} packs)...')

all_sweep_results = {}   # {pack_id: [SimResult, ...]}
all_df_sweep      = {}   # {pack_id: DataFrame}

for _pid in COMPARE_PACK_IDS:
    if _pid not in all_results:
        continue
    _sw = temperature_sweep(
        pack=db.packs[_pid], mission=mission, uav=uav,
        discharge_pts=db.discharge_pts,
        temperatures_c=TEMP_SWEEP, dt_s=5.0
    )
    all_sweep_results[_pid] = _sw
    all_df_sweep[_pid] = pd.DataFrame([{
        'Ambient (C)':   t,
        'Final SoC (%)': round(r.final_soc, 1),
        'Peak sag (V)':  round(r.peak_sag_v, 3),
        'Min V (V)':     round(r.min_voltage, 3),
        'Max T (\u00b0C)':  round(r.max_temp_c, 1),
        'Depleted':      r.depleted}
        for t, r in zip(TEMP_SWEEP, _sw)])
    print(f'\n  {_pid}:')
    print(all_df_sweep[_pid].to_string(index=False))

# Primary aliases for backward compatibility
sweep_results = all_sweep_results.get(PRIMARY_PACK_ID,
                    next(iter(all_sweep_results.values())))
df_sweep      = all_df_sweep.get(PRIMARY_PACK_ID,
                    next(iter(all_df_sweep.values())))
print('\nDone.')

Running temperature sweeps (12 temps x 6 packs)...

  BAT_LE_12S2P:
 Ambient (C)  Final SoC (%)  Peak sag (V)  Min V (V)  Max T (°C)  Depleted
         -10           54.2         3.005     42.400        -7.3     False
          -5           57.4         2.465     43.083        -2.8     False
           0           60.1         2.048     43.645         1.8     False
           5           62.1         1.721     44.084         6.5     False
          10           63.8         1.462     44.450        11.3     False
          15           64.9         1.254     44.705        16.1     False
          20           65.9         1.084     44.928        21.0     False
          25           66.6         0.945     45.093        25.9     False
          30           66.6         0.830     45.142        30.8     False
          35           66.0         0.734     45.108        35.7     False
          40           65.5         0.653     45.061        40.6     False
          45           64.7     

## 3 · Flight Log Analysis (optional)

In [5]:
log = fitted = None
if INCLUDE_LOG:
    from batteries.log_importer import generate_synthetic_log
    from batteries.parameter_fitter import fit_all, apply_fitted_params
    print('Generating synthetic flight log and fitting parameters...')
    log = generate_synthetic_log(primary_pack, mission, uav, db.discharge_pts,
                                  ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0,
                                  noise_v=0.03, noise_i=0.8)
    fitted = fit_all(log, primary_pack.pack_capacity_ah,
                     primary_pack.pack_voltage_cutoff,
                     chem_id=primary_pack.chemistry_id,
                     pack_id=primary_pack.battery_id)
    print(fitted.summary())
else:
    print('Skipping log analysis (INCLUDE_LOG=False)')

Generating synthetic flight log and fitting parameters...
  [1/5] Fitting internal resistance...
        R_internal_mohm: 10.7980 ± 18.6470  R²=0.255  n=706
  [2/5] Reconstructing OCV curve...
        12 OCV points  R²=0.298
  [3/5] Fitting Peukert exponent...
        peukert_k: 1.0233 ± 0.0051  R²=0.000  n=725
  [4/5] Fitting Arrhenius temperature coefficients...
        B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
        B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  [5/5] Estimating actual capacity...
        actual_capacity_ah: 8.0000 ± 0.0000  R²=0.000  n=0
══ Fitted Battery Parameters ══
  R_internal   : R_internal_mohm: 10.7980 ± 18.6470  R²=0.255  n=706
  Capacity     : actual_capacity_ah: 8.0000 ± 0.0000  R²=0.000  n=0
  Peukert k    : peukert_k: 1.0233 ± 0.0051  R²=0.000  n=725
  B_ohmic      : B_ohmic_K: 0.0000 ± 0.0000  R²=0.000  n=0
  B_ct         : B_ct_K: 0.0000 ± 0.0000  R²=0.000  n=0
  OCV curve    : 12 points fitted
  ⚠  IR regression R²=0.26 — low confidence (<0.5). Consider

## 4 · Generate Excel Report

In [6]:
print(f'Generating report: {OUT_PATH}')
report_path = generate_report(
    out_path=OUT_PATH,
    results=compare_results,
    packs=compare_packs,
    mission=mission,
    uav_name=UAV_ID,
    ambient_temp_c=AMBIENT_TEMP_C,
    temp_sweep_temps=TEMP_SWEEP,
    temp_sweep_results=sweep_results,
    flight_log=log,
    fitted_params=fitted,
    primary_pack=primary_pack,
)
print(f'Report saved: {report_path}')

Generating report: UAV_Battery_Report.xlsx
✓ Report saved → UAV_Battery_Report.xlsx
Report saved: UAV_Battery_Report.xlsx


## 4b · Generate PDF Summary Report

A beautiful multi-page PDF with cover, mission timeline, battery scorecard, temperature sweep, and simulation charts.

In [7]:
import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────
PDF_OUT_PATH  = 'UAV_Battery_Report.pdf'
COMPANY_NAME  = 'Pen Aviation Sdn Bhd'

PHASE_COLORS = {
    'IDLE':            '#AAAAAA',
    'TAKEOFF':         '#FF9944',
    'CLIMB':           '#FFCC44',
    'CRUISE':          '#44AA66',
    'HOVER':           '#4488FF',
    'DESCEND':         '#88AADD',
    'LAND':            '#CC88DD',
    'PAYLOAD_OPS':     '#FF6688',
    'EMERGENCY':       '#FF2222',
    'VTOL_TRANSITION': '#FF6611',
    'VTOL_HOVER':      '#22AAFF',
    'FW_CRUISE':       '#00CC77',
    'FW_CLIMB':        '#AACC44',
    'FW_DESCEND':      '#99CCEE',
}
CHEM_COLORS  = {'LIPO':'#FFB347','LION21':'#4488FF','LIFEPO4':'#66BB6A',
                 'SSS':'#AB47BC','LITO':'#26C6DA','LION':'#FF7043','LIHV':'#FF6E40'}

DARK  = '#1A237E'
ACCENT= '#2196F3'
LIGHT = '#F5F7FA'
TEXT  = '#212121'

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.grid': True,
                     'grid.alpha': 0.2, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})

def _header(fig, title, subtitle='', page_num=None):
    ax_h = fig.add_axes([0, 0.93, 1, 0.07])
    ax_h.set_facecolor(DARK); ax_h.set_xlim(0,1); ax_h.set_ylim(0,1)
    ax_h.axis('off')
    ax_h.text(0.02, 0.55, title, color='white', fontsize=14, fontweight='bold', va='center')
    if subtitle:
        ax_h.text(0.02, 0.15, subtitle, color='#90CAF9', fontsize=9, va='center')
    if page_num:
        ax_h.text(0.98, 0.5, str(page_num), color='#90CAF9', fontsize=9,
                  ha='right', va='center')
    ax_h.axhline(0, color=ACCENT, linewidth=3)

def _footer(fig, left='', right=''):
    ax_f = fig.add_axes([0, 0, 1, 0.035])
    ax_f.set_facecolor('#ECEFF1'); ax_f.set_xlim(0,1); ax_f.set_ylim(0,1)
    ax_f.axis('off')
    ax_f.text(0.02, 0.5, left,  color='#607D8B', fontsize=7, va='center')
    ax_f.text(0.98, 0.5, right, color='#607D8B', fontsize=7, va='center', ha='right')

def _pill(ax, x, y, text, color, width=0.18, height=0.08):
    from matplotlib.patches import FancyBboxPatch
    box = FancyBboxPatch((x-width/2, y-height/2), width, height,
                          boxstyle='round,pad=0.01', facecolor=color,
                          edgecolor='none', transform=ax.transAxes, clip_on=False,
                          zorder=5)
    ax.add_patch(box)
    ax.text(x, y, text, transform=ax.transAxes, ha='center', va='center',
            color='white', fontsize=8, fontweight='bold', zorder=6)

now_str  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
run_info = (f'Mission: {MISSION_ID}  |  UAV: {UAV_ID}  |  '
            f'Temp: {AMBIENT_TEMP_C}°C  |  Generated: {now_str}')

with PdfPages(PDF_OUT_PATH) as pdf:

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 1 — COVER
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    fig.patch.set_facecolor(DARK)

    fig.text(0.5, 0.72, 'UAV Battery Analysis', color='white',
             fontsize=38, fontweight='bold', ha='center', va='center')
    fig.text(0.5, 0.62, 'Performance, Simulation & Scorecard Report',
             color='#90CAF9', fontsize=18, ha='center', va='center')

    ax_bar = fig.add_axes([0.15, 0.56, 0.70, 0.004])
    ax_bar.set_facecolor(ACCENT); ax_bar.axis('off')

    meta = (f'{COMPANY_NAME}\n\n'
            f'Mission          {MISSION_ID}\n'
            f'UAV config       {UAV_ID}\n'
            f'Primary pack     {PRIMARY_PACK_ID}\n'
            f'Temperature      {AMBIENT_TEMP_C}°C\n'
            f'Generated        {now_str}')
    fig.text(0.5, 0.35, meta, color='#CFD8DC', fontsize=12,
             ha='center', va='center', linespacing=1.8,
             fontfamily='monospace')

    ax_pill = fig.add_axes([0.35, 0.08, 0.30, 0.10])
    ax_pill.set_facecolor(ACCENT); ax_pill.axis('off')
    ax_pill.text(0.5, 0.6, f'{len(compare_packs)} batteries analysed',
                 color='white', fontsize=14, fontweight='bold',
                 ha='center', va='center', transform=ax_pill.transAxes)
    ax_pill.text(0.5, 0.2, f'across {len(TEMP_SWEEP)} temperature points',
                 color='#E3F2FD', fontsize=10,
                 ha='center', va='center', transform=ax_pill.transAxes)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 2 — MISSION PROFILE + EQUIPMENT SUMMARY
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Mission Profile', run_info, 'p.2')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs = gridspec.GridSpec(2, 2, figure=fig, left=0.06, right=0.97,
                           top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)

    ax_gantt = fig.add_subplot(gs[0, :])
    x_pos = 0
    for ph in mission.phases:
        color = PHASE_COLORS.get(ph.phase_type, '#CCC')
        ax_gantt.barh(0, ph.duration_s, left=x_pos, height=0.5,
                      color=color, edgecolor='white', linewidth=0.5)
        if ph.duration_s > 30:
            ax_gantt.text(x_pos + ph.duration_s/2, 0,
                          ph.phase_type + '\n' + str(int(ph.duration_s)) + 's',
                          ha='center', va='center', fontsize=8,
                          fontweight='bold', color='white')
        x_pos += ph.duration_s
    ax_gantt.set_xlim(0, x_pos); ax_gantt.set_yticks([])
    ax_gantt.set_xlabel('Time (s)'); ax_gantt.set_title('Mission Phase Timeline')

    ax_pie = fig.add_subplot(gs[1, 0])
    durations = [ph.duration_s for ph in mission.phases]
    labels    = [ph.phase_type for ph in mission.phases]
    colors_p  = [PHASE_COLORS.get(l, '#888') for l in labels]
    ax_pie.pie(durations, labels=labels, colors=colors_p, autopct='%1.0f%%',
               startangle=90, textprops={'fontsize': 8})
    ax_pie.set_title('Phase breakdown')

    ax_sv = fig.add_subplot(gs[1, 1])
    import matplotlib.cm as _cm2
    _cmap2 = _cm2.get_cmap('tab10', max(len(compare_results), 1))
    ax2_v = ax_sv.twinx()
    for _ri2, (_r2, _p2) in enumerate(zip(compare_results, compare_packs)):
        _col2 = _cmap2(_ri2)
        _t2 = np.array(_r2.time_s)
        _lbl2 = _p2.battery_id[:22]
        ax_sv.plot(_t2, _r2.soc_pct, color=_col2, linewidth=1.5, label=_lbl2)
        ax2_v.plot(_t2, _r2.voltage_v, color=_col2, linewidth=1.0,
                   linestyle='--', alpha=0.7)
    ax_sv.set_xlabel('Time (s)'); ax_sv.set_ylabel('SoC (%)')
    ax2_v.set_ylabel('Voltage (V)')
    ax_sv.set_title(f'SoC & Voltage — {len(compare_results)} packs')
    ax_sv.set_ylim(0, 110)
    if len(compare_results) <= 8:
        ax_sv.legend(fontsize=7, loc='lower left')

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 3 — BATTERY SCORECARD
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Battery Scorecard', run_info, 'p.3')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    ax_sc = fig.add_axes([0.04, 0.08, 0.92, 0.80])
    ax_sc.axis('off')

    col_labels = ['Battery', 'Chemistry', 'Energy\n(Wh)', 'Weight\n(g)',
                  'Wh/kg', 'Final SoC\n(%)', 'Min V\n(V)',
                  'Peak sag\n(V)', 'Max I\n(A)', 'Status']
    col_widths = [0.18, 0.09, 0.07, 0.07, 0.06, 0.09, 0.07, 0.08, 0.07, 0.09]

    y = 0.95; row_h = 0.075
    x = 0.0
    for lbl, w in zip(col_labels, col_widths):
        ax_sc.text(x + w/2, y, lbl, ha='center', va='center', fontsize=8,
                   fontweight='bold', color='white',
                   bbox=dict(boxstyle='square,pad=0.3', facecolor=DARK, edgecolor='none'))
        x += w

    for idx, (r, p) in enumerate(zip(compare_results, compare_packs)):
        y -= row_h
        bg = '#FAFAFA' if idx % 2 == 0 else 'white'
        status = 'PASS' if not r.depleted and r.final_soc > 15 else \
                 ('MARGINAL' if not r.depleted else 'FAIL')
        status_color = {'PASS': '#2E7D32', 'MARGINAL': '#F57F17', 'FAIL': '#C62828'}[status]
        vals = [p.battery_id, p.chemistry_id, f'{p.pack_energy_wh:.0f}',
                f'{p.pack_weight_g:.0f}',
                f'{getattr(p, "specific_energy_wh_kg", p.pack_energy_wh/p.pack_weight_g*1000):.0f}',
                f'{r.final_soc:.1f}', f'{r.min_voltage:.3f}',
                f'{r.peak_sag_v:.3f}', f'{r.max_current:.1f}', status]
        x = 0.0
        for v, w in zip(vals, col_widths):
            color = status_color if v == status else TEXT
            fw    = 'bold' if v == status else 'normal'
            ax_sc.text(x + w/2, y + row_h/2, v, ha='center', va='center',
                       fontsize=8, color=color, fontweight=fw,
                       bbox=dict(boxstyle='square,pad=0.3',
                                 facecolor=bg, edgecolor='#E0E0E0', linewidth=0.4))
            x += w

    legend_patches = [mpatches.Patch(facecolor=color, label=label, alpha=0.85)
                      for label, color in [('PASS','#2E7D32'),('MARGINAL','#F57F17'),('FAIL','#C62828')]]
    ax_sc.legend(handles=legend_patches, loc='lower right', fontsize=8, title='Status', framealpha=0.9)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 4 — TEMPERATURE SENSITIVITY
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Temperature Sensitivity', run_info, 'p.4')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs4 = gridspec.GridSpec(2, 2, figure=fig, left=0.08, right=0.96,
                            top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)

    import matplotlib.cm as _cm4
    _cmap4 = _cm4.get_cmap('tab10', max(len(compare_packs), 1))

    _axes4 = [fig.add_subplot(gs4[_r4, _c4])
              for _r4, _c4 in [(0,0),(0,1),(1,0),(1,1)]]
    _metrics4 = [
        ('Final SoC (%)',  'Final SoC (%)'),
        ('Peak sag (V)',   'Peak sag (V)'),
        ('Min V (V)',      'Min V (V)'),
        ('_self_heat',     'Self-heating (°C)'),
    ]

    for _pidx4, (_pid4, _df4) in enumerate(all_df_sweep.items()):
        _col4 = _cmap4(_pidx4)
        _lbl4 = _pid4[:22]
        _t4   = _df4['Ambient (C)'].values
        _dep4 = _df4['Depleted'].values
        for _ax4, (_field4, _ylabel4) in zip(_axes4, _metrics4):
            _y4 = (_df4['Max T (°C)'].values - _t4
                   if _field4 == '_self_heat' else _df4[_field4].values)
            _ax4.plot(_t4, _y4, color=_col4, linewidth=1.8, label=_lbl4)
            _ax4.scatter(_t4, _y4,
                         c=['#E53935' if d else _col4 for d in _dep4],
                         s=30, zorder=4)

    for _ax4, (_, _ylabel4) in zip(_axes4, _metrics4):
        _ax4.set_xlabel('Ambient (°C)'); _ax4.set_ylabel(_ylabel4)
        _ax4.set_title(_ylabel4)

    if len(compare_packs) <= 6:
        _axes4[0].legend(fontsize=7, loc='best')
    else:
        _handles4 = [mpatches.Patch(color=_cmap4(_ii4), label=_pp4.battery_id[:22])
                     for _ii4, _pp4 in enumerate(compare_packs)]
        fig.legend(handles=_handles4, loc='lower center',
                   ncol=min(len(compare_packs), 4), fontsize=7,
                   bbox_to_anchor=(0.5, 0.0))
    _dep4_leg = [mpatches.Patch(color=ACCENT, label='Completed'),
                 mpatches.Patch(color='#E53935', label='Depleted')]
    fig.legend(handles=_dep4_leg, loc='lower right', ncol=2, fontsize=8)

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 5 — MULTI-BATTERY COMPARISON CHARTS
    # ══════════════════════════════════════════════════════════════════════════
    fig = plt.figure(figsize=(11.69, 8.27))
    _header(fig, 'Battery Comparison Curves', run_info, 'p.5')
    _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

    gs5 = gridspec.GridSpec(2, 2, figure=fig, left=0.08, right=0.96,
                            top=0.89, bottom=0.08, hspace=0.45, wspace=0.35)
    import matplotlib.cm as _cm5
    _cmap5 = _cm5.get_cmap('tab10', max(len(compare_results), 1))
    _pal5  = [_cmap5(j) for j in range(len(compare_results))]

    axes5 = [fig.add_subplot(gs5[r, c]) for r in range(2) for c in range(2)]
    fields = [('soc_pct', 'SoC (%)'), ('voltage_v', 'Voltage (V)'),
              ('current_a', 'Current (A)'), ('temp_c', 'Temp (°C)')]

    for (field, ylabel), ax in zip(fields, axes5):
        for r, p, col in zip(compare_results, compare_packs, _pal5):
            t_r = np.array(r.time_s)
            y_r = np.array(getattr(r, field))
            ax.plot(t_r, y_r, color=col, linewidth=1.8,
                    label=f'{p.battery_id} ({p.pack_energy_wh:.0f}Wh)')
        ax.set_xlabel('Time (s)'); ax.set_ylabel(ylabel); ax.set_title(ylabel)

    axes5[0].legend(fontsize=7, loc='lower left')

    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ══════════════════════════════════════════════════════════════════════════
    # PAGE 6 — FITTED PARAMETERS (if log available)
    # ══════════════════════════════════════════════════════════════════════════
    if fitted:
        fig = plt.figure(figsize=(11.69, 8.27))
        _header(fig, 'Reverse-Engineered Battery Parameters', run_info, 'p.6')
        _footer(fig, COMPANY_NAME, PDF_OUT_PATH)

        ax_fp = fig.add_axes([0.05, 0.10, 0.90, 0.78])
        ax_fp.axis('off')

        rows_fp = [
            ('Parameter', 'Fitted value', 'Catalog value', 'Δ vs catalog', 'R²', 'Confidence'),
            ('R_internal (mΩ)',
             f'{fitted.r_internal_mohm.value:.2f} ± {fitted.r_internal_mohm.uncertainty:.2f}' if fitted.r_internal_mohm else '—',
             f'{primary_pack.internal_resistance_mohm:.1f}',
             f'{fitted.r_internal_mohm.value - primary_pack.internal_resistance_mohm:+.1f}' if fitted.r_internal_mohm else '—',
             f'{fitted.r_internal_mohm.r_squared:.3f}' if fitted.r_internal_mohm else '—',
             'Good' if fitted.r_internal_mohm and fitted.r_internal_mohm.r_squared > 0.5 else 'Low'),
            ('Peukert k',
             f'{fitted.peukert_k.value:.4f}' if fitted.peukert_k else '—',
             '1.050', '—', '—', '—'),
            ('Capacity (Ah)',
             f'{fitted.actual_capacity_ah.value:.3f}' if fitted.actual_capacity_ah else '—',
             f'{primary_pack.pack_capacity_ah:.3f}',
             f'{fitted.actual_capacity_ah.value - primary_pack.pack_capacity_ah:+.3f}' if fitted.actual_capacity_ah else '—',
             '—', '—'),
            ('Degradation', f'{fitted.degradation_pct:.1f}%', '0%', '—', '—', '—'),
        ]

        col_x = [0.0, 0.25, 0.42, 0.57, 0.70, 0.80]
        col_w = [0.25, 0.17, 0.15, 0.13, 0.10, 0.20]
        y_fp = 0.92; row_fp = 0.11

        for ridx, row_data in enumerate(rows_fp):
            bg = DARK if ridx == 0 else ('#F5F5F5' if ridx % 2 == 0 else 'white')
            fc = 'white' if ridx == 0 else TEXT
            fw = 'bold' if ridx == 0 else 'normal'
            for cidx, (val, x, w) in enumerate(zip(row_data, col_x, col_w)):
                if ridx > 0 and cidx == 5:
                    fc2 = '#2E7D32' if val == 'Good' else ('#F57F17' if val == 'Low' else TEXT)
                else:
                    fc2 = fc
                ax_fp.text(x + w/2, y_fp - ridx * row_fp + row_fp/2,
                           val, ha='center', va='center', fontsize=9,
                           color=fc2, fontweight=fw,
                           bbox=dict(boxstyle='square,pad=0.3',
                                     facecolor=bg, edgecolor='#E0E0E0',
                                     linewidth=0.5))

        if fitted.fit_warnings:
            wtext = '  ⚠  ' + '   ⚠  '.join(fitted.fit_warnings)
            ax_fp.text(0.5, 0.05, wtext, ha='center', va='center',
                       fontsize=8, color='#E65100',
                       bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF3E0',
                                 edgecolor='#FF9800'))

        pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

print(f'PDF report saved: {PDF_OUT_PATH}')
print(f'Pages: cover + mission + scorecard + temperature + comparison' +
      (' + fitted params' if fitted else ''))

PDF report saved: UAV_Battery_Report.pdf
Pages: cover + mission + scorecard + temperature + comparison + fitted params


## 5 · Quick Battery Selection Table

Which batteries from the catalog can safely complete this mission at all temperature sweep points?

In [8]:
from mission.simulator import SimulationResult
sel_packs = [db.packs[pid] for pid in COMPARE_PACK_IDS if pid in db.packs]
selection_rows = []
print(f'Testing {len(sel_packs)} selected packs across {len(TEMP_SWEEP)} temperatures...')
for p in sel_packs:
    row = {'Pack_ID': p.battery_id, 'Chemistry': p.chemistry_id,
           'Energy (Wh)': p.pack_energy_wh, 'Weight (g)': p.pack_weight_g,
           'Wh/kg': p.specific_energy_wh_kg}
    pass_all = True
    for temp in TEMP_SWEEP:
        try:
            r = run_simulation(p, mission, uav, db.discharge_pts,
                               ambient_temp_c=temp, dt_s=10.0)
            row[f'{temp}C SoC'] = round(r.final_soc, 1)
            row[f'{temp}C pass'] = not r.depleted and r.final_soc > 15
            if r.depleted or r.final_soc <= 15: pass_all = False
        except Exception as e:
            row[f'{temp}C SoC'] = 'ERR'; row[f'{temp}C pass'] = False
            pass_all = False
    row['All temps PASS'] = pass_all
    selection_rows.append(row)

df_sel = pd.DataFrame(selection_rows).set_index('Pack_ID')
df_sel.sort_values('All temps PASS', ascending=False)

Testing 6 selected packs across 12 temperatures...


,Chemistry,Energy (Wh),Weight (g),Wh/kg,-10C SoC,-10C pass,-5C SoC,-5C pass,0C SoC,0C pass,...,25C pass,30C SoC,30C pass,35C SoC,35C pass,40C SoC,40C pass,45C SoC,45C pass,All temps PASS
Pack_ID,,,,,,,,,,,,,,,,,,,,,
BAT_LE_12S2P,LION21,350.0,1656.0,212.0,54.3,True,57.4,True,60.1,True,...,True,66.6,True,66.0,True,65.5,True,64.7,True,True
GREPOW_TATTU_12S_LIPO_DRONE_BATTERY,LIPO,444.0,2800.0,158.6,62.9,True,65.8,True,68.2,True,...,True,74.0,True,73.8,True,73.5,True,73.0,True,True
GREPOW_TATTU_12S_10000MAH_44_4V_30C_LIPO_BATTERY,LIPO,444.0,2800.0,158.6,62.9,True,65.8,True,68.2,True,...,True,74.0,True,73.8,True,73.5,True,73.0,True,True
TATTU_TATTU_PLUS_1_0_COMPACT_VERSION_10000MAH_44_4,LIPO,444.0,2978.0,149.1,62.9,True,65.8,True,68.2,True,...,True,74.0,True,73.8,True,73.5,True,73.0,True,True
GREPOW_TATTU_PLUS_1_0_12S_10000MAH_44_4V_15C_LIPO_,LIPO,444.0,2978.0,149.1,62.9,True,65.8,True,68.2,True,...,True,74.0,True,73.8,True,73.5,True,73.0,True,True
TATTU_TATTU_SEMI_SOLID_STATE_330WH_KG_25000MAH_10C,LIPO,1110.0,3788.0,293.0,86.3,True,87.3,True,88.2,True,...,True,90.3,True,90.2,True,90.1,True,90.0,True,True
